# Works and temporal perspective

Relate sentence-level detection outcomes to the referenced works and their temporal perspective.

In [ ]:
from shared import paths
from IPython.display import display


## Setup

In [ ]:
paths.prepare_directories()


## Attach annotation attributes

In [ ]:
import pandas as pd
import re
import ast
from shared.utils import load_pickle

PREDICTIONS_CSV = str(paths.sentence_run(3) / 'general/all_folds_predictions.csv')
OUTPUT_CSV = str(paths.sentence_run(3) / 'general/all_folds_predictions_with_meta.csv')
DIVIDER = '_' * 60
ABBREVIATIONS = [
    'Mr',
    'Mrs',
    'Ms',
    'Dr',
    'Prof',
    'Sr',
    'Jr',
    'St',
    'vs',
    'i.e',
    'e.g',
    'No',
    'Dept',
    'Inc',
    'Ltd',
    'Jan',
    'Feb',
    'Mar',
    'Apr',
    'Jun',
    'Jul',
    'Aug',
    'Sep',
    'Sept',
    'Oct',
    'Nov',
    'Dec',
    'Vol',
]
from shared.text import protect_abbreviations
from shared.text import restore_abbreviations


def extract_meta():
    letters = load_pickle()
    reviewed_letters = [l for l in letters if getattr(l, 'reviewed', False) and l.intertextual_traces]
    meta_map = dict()
    for letter in reviewed_letters:
        anno_text = letter.anno_text
        match = re.search(f'{DIVIDER}\\n\\n(.*?)\\n\\n{DIVIDER}', anno_text, re.DOTALL)
        if not match:
            continue
        rel_text = match.group(1)
        rel_start = match.start(1)
        rel_text = protect_abbreviations(rel_text, ABBREVIATIONS)
        sentence_pattern = re.compile('.{6,}?[.!?;](?:["\\\'*]*)?(?:\\[\\[\\d+\\]\\])?(?=\\s|$)', re.DOTALL)
        segments = sentence_pattern.findall(rel_text)
        remainder = rel_text[sum((len(s) for s in segments)):].strip()
        if remainder:
            segments.append(remainder)
        sentences = [restore_abbreviations(s) for s in segments]
        sentences = [re.sub('\\[\\[\\d+\\]\\]$', '', s).strip() for s in sentences if s.strip()]
        offset = rel_start
        sentence_data = []
        for sent in sentences:
            sent_start = anno_text.find(sent, offset)
            if sent_start == -1:
                continue
            sent_end = sent_start + len(sent)
            offset = sent_end
            sentence_data.append((sent_start, sent_end, sent))
        for i, (sent_start, sent_end, _) in enumerate(sentence_data):
            pairs = set()
            for trace in letter.intertextual_traces:
                for ann in trace.annotations:
                    if not ann.category:
                        continue
                    if ann.end_offset <= sent_start or ann.start_offset >= sent_end:
                        continue
                    novels = ann.target_novel
                    if isinstance(novels, str):
                        novels = [novels]
                    elif not isinstance(novels, list):
                        novels = [None]
                    temps = ann.temporal_perspective
                    if isinstance(temps, str):
                        temps = [temps]
                    elif not isinstance(temps, list):
                        temps = [None]
                    for nv in novels:
                        for tp in temps:
                            if nv or tp:
                                pairs.add((nv, tp))
            meta_map[letter.id, i] = [{'novel': nv, 'temporal': tp} for nv, tp in sorted(pairs)]
    return meta_map


def main():
    print(' Loading predictions...')
    df = pd.read_csv(PREDICTIONS_CSV)
    for col in ['true_labels', 'pred_labels']:
        if col in df:
            df[col] = df[col].apply(ast.literal_eval)
    print(' Computing binary labels...')
    df['true_binary'] = df['true_labels'].apply(lambda x: int(any((v == 1 for v in x))))
    df['pred_binary'] = df['pred_labels'].apply(lambda x: int(any((v == 1 for v in x))))
    print(' Extracting work and temporal-perspective pairs...')
    meta_map = extract_meta()
    print(' Joining predictions and annotations...')
    df['novel_temporal_pairs'] = df.apply(lambda row: meta_map.get((row['letter_id'], row['sentence_id']), []), axis=1)
    print(f' Saving CSV to:\n{OUTPUT_CSV}')
    df.to_csv(OUTPUT_CSV, index=False)
    print(' Done.')
if __name__ == '__main__':
    main()


## Reference detection

In [ ]:
import pandas as pd
import ast
from sklearn.metrics import precision_recall_fscore_support

CSV_PATH = str(paths.sentence_run(3) / 'general/all_folds_predictions_with_meta.csv')
df = pd.read_csv(CSV_PATH)
df['novel_temporal_pairs'] = df['novel_temporal_pairs'].apply(ast.literal_eval)


def evaluate(y_true, y_pred):
    p, r, f1, s = precision_recall_fscore_support(y_true, y_pred, average='binary', zero_division=0)
    return {
        'Support': int(sum(y_true)),
        'Precision': round(p, 3),
        'Recall': round(r, 3),
        'F1': round(f1, 3),
    }
stats = {}
for idx, row in df.iterrows():
    for pair in row['novel_temporal_pairs']:
        novel = pair.get('novel', 'UNKNOWN')
        temp = pair.get('temporal', 'UNKNOWN')
        if novel not in stats:
            stats[novel] = {'All': {'y_true': [], 'y_pred': []}}
        stats[novel]['All']['y_true'].append(row['true_binary'])
        stats[novel]['All']['y_pred'].append(row['pred_binary'])
        if temp not in stats[novel]:
            stats[novel][temp] = {'y_true': [], 'y_pred': []}
        stats[novel][temp]['y_true'].append(row['true_binary'])
        stats[novel][temp]['y_pred'].append(row['pred_binary'])
for novel, temp_dict in stats.items():
    print(f'\n=== {novel} ===')
    rows = []
    for temp, vals in temp_dict.items():
        if len(vals['y_true']) > 0:
            metrics = evaluate(vals['y_true'], vals['y_pred'])
            rows.append(
                [temp, metrics['Support'], metrics['Precision'], metrics['Recall'], metrics['F1']],
            )
    df_out = pd.DataFrame(rows, columns=['Temporal', 'Support', 'Precision', 'Recall', 'F1'])
    print(df_out.to_string(index=False))


## Recall by work and temporal perspective

In [ ]:
import pandas as pd
import numpy as np

TEMP_KEY_MAP = {'analeptic': 'Analeptic', 'current_work': 'Current', 'proleptic': 'Proleptic', 'All': 'All'}
YEAR_MAP = {
    'pickwick_papers': 1836,
    'oliver_twist': 1837,
    'nicholas_nickleby': 1838,
    'old_curiosity_shop': 1840,
    'barnaby_rudge': 1841,
    'martin_chuzzlewit': 1843,
    'dombey_and_son': 1848,
    'david_copperfield': 1850,
    'bleak_house': 1853,
    'hard_times': 1854,
    'little_dorrit': 1857,
    'a_tale_of_two_cities': 1859,
    'great_expectations': 1861,
    'our_mutual_friend': 1865,
    'edwin_drood': 1870,
    'a_christmas_carol': 1843,
    'a_childs_history_of_england': 1853,
    'american_notes': 1842,
    'pictures_of_italy': 1846,
    'memoirs_of_joseph_grimaldi': 1838,
    'sketches_by_boz': 1836,
    'master_humphreys_clock': 1840,
    'is_she_his_wife': 1836,
    'the_village_coquettes': 1836,
    'others': 9997,
}
TITLE_MAP = {
    'pickwick_papers': 'Pickwick Papers',
    'oliver_twist': 'Oliver Twist',
    'nicholas_nickleby': 'Nicholas Nickleby',
    'old_curiosity_shop': 'The Old Curiosity Shop',
    'barnaby_rudge': 'Barnaby Rudge',
    'martin_chuzzlewit': 'Martin Chuzzlewit',
    'dombey_and_son': 'Dombey And Son',
    'david_copperfield': 'David Copperfield',
    'bleak_house': 'Bleak House',
    'hard_times': 'Hard Times',
    'little_dorrit': 'Little Dorrit',
    'a_tale_of_two_cities': 'A Tale Of Two Cities',
    'great_expectations': 'Great Expectations',
    'our_mutual_friend': 'Our Mutual Friend',
    'edwin_drood': 'The Mystery of Edwin Drood',
    'a_christmas_carol': 'A Christmas Carol',
    'a_childs_history_of_england': 'A Childs History Of England',
    'american_notes': 'American Notes',
    'pictures_of_italy': 'Pictures Of Italy',
    'memoirs_of_joseph_grimaldi': 'Memoirs Of Joseph Grimaldi',
    'sketches_by_boz': 'Sketches By Boz',
    'master_humphreys_clock': 'Master Humphreys Clock',
    'is_she_his_wife': 'Is She His Wife',
    'the_village_coquettes': 'The Village Coquettes',
    'others': 'Others',
}
CATEGORY_MAP = {
    'pickwick_papers': 'Novels and Novellas',
    'oliver_twist': 'Novels and Novellas',
    'nicholas_nickleby': 'Novels and Novellas',
    'old_curiosity_shop': 'Novels and Novellas',
    'barnaby_rudge': 'Novels and Novellas',
    'martin_chuzzlewit': 'Novels and Novellas',
    'dombey_and_son': 'Novels and Novellas',
    'david_copperfield': 'Novels and Novellas',
    'bleak_house': 'Novels and Novellas',
    'hard_times': 'Novels and Novellas',
    'little_dorrit': 'Novels and Novellas',
    'a_tale_of_two_cities': 'Novels and Novellas',
    'great_expectations': 'Novels and Novellas',
    'our_mutual_friend': 'Novels and Novellas',
    'edwin_drood': 'Novels and Novellas',
    'a_christmas_carol': 'Christmas Books',
    'the_chimes': 'Christmas Books',
    'the_cricket_on_the_hearth': 'Christmas Books',
    'the_battle_of_life': 'Christmas Books',
    'the_haunted_man_and_the_ghosts_bargain': 'Christmas Books',
    'sketches_by_boz': 'Short Story Collections',
    'master_humphreys_clock': 'Short Story Collections',
    'sketches_of_young_couples': 'Short Story Collections',
    'is_she_his_wife': 'Short Stories',
    'horatio_sparkins': 'Short Stories',
    'the_lamplighter': 'Short Stories',
    'the_strange_gentleman': 'Plays',
    'the_village_coquettes': 'Plays',
    'a_childs_history_of_england': 'Nonfiction',
    'american_notes': 'Nonfiction',
    'pictures_of_italy': 'Nonfiction',
    'memoirs_of_joseph_grimaldi': 'Nonfiction',
    'others': 'Others',
    'general': 'Others',
    'none': 'Others',
}
NORMALIZE_KEYS = {'the_village_coquette': 'the_village_coquettes', 'general': 'others', 'none': 'others'}


def normalize_key(raw_name: str) -> str:
    return NORMALIZE_KEYS.get(raw_name, raw_name)


def pretty_title(raw_name: str) -> str:
    return TITLE_MAP.get(raw_name, raw_name.replace('_', ' ').title())


def work_year(raw_name: str) -> int:
    return YEAR_MAP.get(raw_name, 9999)


def work_category(raw_name: str) -> str:
    return CATEGORY_MAP.get(raw_name, 'Others')
rows = []
for novel_key, temp_dict in stats.items():
    novel_key = normalize_key(novel_key)
    row = {
        'WorkKey': novel_key,
        'Work': pretty_title(novel_key),
        'Category': work_category(novel_key),
        'Year': work_year(novel_key),
    }
    for raw_key, col_name in TEMP_KEY_MAP.items():
        if raw_key in temp_dict and len(temp_dict[raw_key]['y_true']) > 0:
            m = evaluate(temp_dict[raw_key]['y_true'], temp_dict[raw_key]['y_pred'])
            row[col_name] = {'recall': m['Recall'], 'support': m['Support']}
        else:
            row[col_name] = {'recall': None, 'support': 0}
    rows.append(row)
df_all = pd.DataFrame(rows)


def reassign_small_support(row):
    total_support = row['All']['support']
    if total_support < 10:
        row['Work'] = 'Others'
        row['Category'] = 'Others'
        row['Year'] = 9997
        row['WorkKey'] = 'others'
    return row
df_all = df_all.apply(reassign_small_support, axis=1)
agg_funcs = {}
for col in TEMP_KEY_MAP.values():

    def make_agg(col_name=col):

        def agg(series):
            recalls = []
            total_support = 0
            for v in series:
                if v['support'] > 0:
                    recalls.append(v['recall'])
                    total_support += v['support']
            if total_support == 0:
                return {'recall': None, 'support': 0}
            return {'recall': np.mean(recalls), 'support': total_support}
        return agg
    agg_funcs[col] = make_agg()
df_grouped = df_all.groupby(['Category', 'Work', 'Year'], as_index=False).agg(agg_funcs)


def format_cell(cell):
    if cell['support'] == 0 or cell['recall'] is None:
        return '-'
    return f"{cell['recall']:.2f} ({cell['support']})"
for col in TEMP_KEY_MAP.values():
    df_grouped[col] = df_grouped[col].apply(format_cell)
df_grouped = df_grouped.sort_values(['Category', 'Year', 'Work']).reset_index(drop=True)
summary_rows = []
for cat, group in df_grouped.groupby('Category', sort=False):
    summary = {'Category': cat, 'Work': 'Avg. / Total', 'Year': 9999}
    for col in TEMP_KEY_MAP.values():
        recalls, total_support = ([], 0)
        for cell in group[col]:
            if cell != '-' and '(' in cell:
                recall_str, support_str = cell.split('(')
                recall = float(recall_str.strip())
                support = int(support_str.strip(') '))
                recalls.append(recall)
                total_support += support
        if total_support > 0:
            summary[col] = f'{np.mean(recalls):.2f} ({total_support})'
        else:
            summary[col] = '-'
    summary_rows.append(summary)
df_with_totals = pd.concat([df_grouped, pd.DataFrame(summary_rows)], ignore_index=True)
print('\n=== Works grouped with <10 reassigned to Others (with Totals) ===')
display(df_with_totals)
